In [ ]:
%pip install langchain langchain-community langchain-openai langgraph
%pip install rank_bm25 tiktoken
%pip install fastapi uvicorn docker
%pip install redis gptcache
%pip install ragas cohere
%pip install streamlit boto3

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from rank_bm25 import BM25Okapi

documents = [
    "Databricks is a unified analytics platform",
    "LangChain helps build LLM applications",
    "FastAPI is used for deploying APIs"
]

tokenized_docs = [doc.split() for doc in documents]

bm25 = BM25Okapi(tokenized_docs)

query = "LLM applications"
tokenized_query = query.split()

scores = bm25.get_scores(tokenized_query)

print(scores)

[0.         1.07876219 0.        ]


In [1]:
from langchain_openai import OpenAIEmbeddings
import numpy as np

embeddings = OpenAIEmbeddings()

docs = ["Databricks platform", "LLM applications", "API deployment"]

# Pure NumPy L2 search (avoids faiss-cpu segfaults on Python 3.13 / macOS ARM)
doc_matrix = np.array([embeddings.embed_query(doc) for doc in docs], dtype=np.float32)
query_vec = np.asarray(embeddings.embed_query("AI applications"), dtype=np.float32)
dists = np.linalg.norm(doc_matrix - query_vec, axis=1)
I = np.argsort(dists)[:2]

print(I)

: 

In [ ]:
from langchain.retrievers import EnsembleRetriever

# Assume you have bm25_retriever and vector_retriever

ensemble = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]
)

In [ ]:
def rrf(rank_lists, k=60):
    scores = {}
    for rlist in rank_lists:
        for rank, doc in enumerate(rlist):
            scores[doc] = scores.get(doc, 0) + 1 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [ ]:
from gptcache import cache
from gptcache.adapter import openai

cache.init()

response = openai.ChatCompletion.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is Databricks?"}]
)

print(response)

In [ ]:
from gptcache import cache
from gptcache.adapter import openai

cache.init()

response = openai.ChatCompletion.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is Databricks?"}]
)

print(response)

In [ ]:
import cohere

co = cohere.Client("YOUR_API_KEY")

docs = ["Databricks platform", "LLM apps", "API deployment"]

response = co.rerank(
    query="AI platform",
    documents=docs
)

print(response)

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/ask")
def ask(query: str):
    return {"response": f"You asked: {query}"}

In [1]:
# ================================
# 🚀 END-TO-END HYBRID RAG DEMO
# ================================

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage
from rank_bm25 import BM25Okapi
import numpy as np
import hashlib

# -------------------------------
# 🔹 1. Sample Enterprise Docs
# -------------------------------
documents = [
    "Databricks is a unified analytics platform for big data and AI.",
    "LangChain helps build applications using large language models.",
    "FastAPI is a modern framework for building APIs with Python.",
    "Hybrid search combines keyword and semantic search techniques.",
    "BM25 is a ranking algorithm used in search engines.",
    "Vector databases store embeddings for semantic retrieval."
]

# -------------------------------
# 🔹 2. Initialize Models
# -------------------------------
llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings()

# -------------------------------
# 🔹 3. BM25 Setup
# -------------------------------
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

# -------------------------------
# 🔹 4. Vector Search Setup (NumPy L2 — same math as FAISS IndexFlatL2)
# -------------------------------
doc_matrix = np.array(
    [embeddings.embed_query(doc) for doc in documents], dtype=np.float32
)

# -------------------------------
# 🔹 5. Simple Cache (Semantic Simulation)
# -------------------------------
cache = {}

def get_cache_key(query):
    return hashlib.md5(query.encode()).hexdigest()

# -------------------------------
# 🔹 6. RRF Fusion
# -------------------------------
def rrf_fusion(bm25_rank, vector_rank, k=60):
    scores = {}
    for rank, idx in enumerate(bm25_rank):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)
    for rank, idx in enumerate(vector_rank):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# -------------------------------
# 🔹 7. Query Rewriting
# -------------------------------
def rewrite_query(query):
    prompt = f"Rewrite this query clearly for better search: {query}"
    return llm.invoke([HumanMessage(content=prompt)]).content

# -------------------------------
# 🔹 8. MAIN PIPELINE
# -------------------------------
def ask_question(user_query):
    
    # ✅ Cache Check
    key = get_cache_key(user_query)
    if key in cache:
        print("⚡ Cache Hit")
        return cache[key]

    print("🔄 Processing...\n")

    # 1. Rewrite Query
    rewritten_query = rewrite_query(user_query)
    print("🔹 Rewritten Query:", rewritten_query)

    # 2. BM25 Search
    tokenized_query = rewritten_query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_rank = np.argsort(bm25_scores)[::-1]

    # 3. Vector Search (NumPy L2; avoids faiss kernel crashes on Py3.13 / macOS ARM)
    q = np.asarray(embeddings.embed_query(rewritten_query), dtype=np.float32)
    dists = np.linalg.norm(doc_matrix - q, axis=1)
    vector_rank = np.argsort(dists)[:3]

    # 4. Hybrid Fusion (RRF)
    fused = rrf_fusion(bm25_rank[:3], vector_rank)

    # 5. Top Documents
    top_docs = [documents[idx] for idx, _ in fused[:3]]

    print("\n🔹 Retrieved Docs:")
    for doc in top_docs:
        print("-", doc)

    # 6. Generate Answer
    context = "\n".join(top_docs)
    final_prompt = f"""
    Answer based on context only:

    Context:
    {context}

    Question:
    {user_query}
    """

    answer = llm.invoke([HumanMessage(content=final_prompt)]).content

    # 7. Basic Evaluation (Relevancy Check)
    relevance_prompt = f"""
    Is the answer relevant to the question?

    Question: {user_query}
    Answer: {answer}

    Answer Yes or No.
    """

    relevance = llm.invoke([HumanMessage(content=relevance_prompt)]).content

    # Cache result
    cache[key] = (answer, relevance)

    return answer, relevance

# -------------------------------
# 🔹 9. RUN DEMO
# -------------------------------
query = input("💬 Ask your question: ")

answer, relevance = ask_question(query)

print("\n✅ FINAL ANSWER:\n", answer)
print("\n📊 Relevance Check:", relevance)

🔄 Processing...

🔹 Rewritten Query: How can I quickly get started with FastAPI?

🔹 Retrieved Docs:
- FastAPI is a modern framework for building APIs with Python.
- Vector databases store embeddings for semantic retrieval.
- LangChain helps build applications using large language models.

✅ FINAL ANSWER:
 FastAPI is a modern framework for building APIs with Python.

📊 Relevance Check: Yes.
